# Insurance Claim Cost Prediction and Fraud Detection
### End‑to‑End Data Science Analysis

**Author:** Marzieh Abbasi  
**Course:** DSC 530 – Data Exploration and Analysis  
**Institution:** Bellevue University

# Executive Summary
End‑to‑end data science workflow for insurance analytics including prediction, fraud detection and clustering.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import r2_score, mean_squared_error, classification_report

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import pickle

sns.set(style='whitegrid')
print('Libraries loaded')

# Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/insurance_claims.csv')
print(df.shape)
df.head()

# Data Cleaning

In [ ]:
df = df.replace('?', np.nan)

for col in df.select_dtypes(include=np.number):
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(exclude=np.number):
    df[col] = df[col].fillna(df[col].mode()[0])

# Feature Engineering

In [ ]:
df['claim_per_month'] = df['total_claim_amount']/(df['months_as_customer']+1)

le = LabelEncoder()

for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col])

# Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['total_claim_amount'], bins=30, kde=True)
plt.title('Claim Amount Distribution')
plt.show()

# Regression Model – Claim Prediction

In [ ]:
y = df['total_claim_amount']
X = df.drop(columns=['total_claim_amount','fraud_reported'])
X = pd.get_dummies(X, drop_first=True)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

rf = RandomForestRegressor(n_estimators=300,random_state=42)
rf.fit(X_train,y_train)

pred = rf.predict(X_test)
print('R2:', r2_score(y_test,pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test,pred)))

# Fraud Detection Model

In [ ]:
X = df.drop(columns=['fraud_reported'])
y = df['fraud_reported']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

clf = RandomForestClassifier(n_estimators=300,random_state=42)
clf.fit(X_train,y_train)

pred = clf.predict(X_test)
print(classification_report(y_test,pred))

# Customer Segmentation

In [ ]:
features = df.select_dtypes(include=np.number)

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(scaled)

pca = PCA(n_components=2)
pca_res = pca.fit_transform(scaled)

plt.scatter(pca_res[:,0],pca_res[:,1],c=clusters)
plt.title('Customer Segmentation')
plt.show()

# Model Persistence

In [ ]:
with open('../models/random_forest_model.pkl','wb') as f:
    pickle.dump(rf,f)